In [1]:
# Bootstrap Local - Setup de diretórios e schemas
import logging
from pathlib import Path
from lakehouse_engine.spark import get_spark_session

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

spark = get_spark_session()
logger.info('Spark Session obtida com sucesso')

data_root = './data/olist'
landing_root = f'{data_root}/landing'

Path(f'{landing_root}/files').mkdir(parents=True, exist_ok=True)
Path(f'{landing_root}/checkpoints').mkdir(parents=True, exist_ok=True)
logger.info(f'Diretórios criados em: {landing_root}')

databases = ['landing', 'bronze', 'silver', 'gold']
for db in databases:
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {db}')
    logger.info(f'Database olist_{db} criado')

tables = [
    'customers',
    'order_items',
    'order_reviews',
    'orders',
    'products',
    'sellers'
]
for table in tables:
    files_path = f'{landing_root}/files/{table}'
    checkpoint_path = f'{landing_root}/checkpoints/{table}'
    Path(files_path).mkdir(parents=True, exist_ok=True)
    Path(checkpoint_path).mkdir(parents=True, exist_ok=True)
    logger.info(f'Diretórios criados para tabela: {table}')

logger.info('=== Bootstrap Local Concluído ===')
logger.info(f'Ambiente pronto para desenvolvimento local')
logger.info(f'Root Data: {data_root}')
logger.info(f'Databases criados: {databases}')
logger.info(f'Tabelas configuradas: {len(tables)} tabelas')

[2026-03-17 13:57:27,085] [WARNING] [LOCAL] Running in local mode.
[2026-03-17 13:57:32,478] [INFO] Spark Session obtida com sucesso
[2026-03-17 13:57:32,479] [INFO] Diretórios criados em: ./data/olist/landing
[2026-03-17 13:57:34,981] [INFO] Database olist_landing criado
[2026-03-17 13:57:35,012] [INFO] Database olist_bronze criado
[2026-03-17 13:57:35,040] [INFO] Database olist_silver criado
[2026-03-17 13:57:35,060] [INFO] Database olist_gold criado
[2026-03-17 13:57:35,061] [INFO] Diretórios criados para tabela: customers
[2026-03-17 13:57:35,062] [INFO] Diretórios criados para tabela: order_items
[2026-03-17 13:57:35,063] [INFO] Diretórios criados para tabela: order_reviews
[2026-03-17 13:57:35,066] [INFO] Diretórios criados para tabela: orders
[2026-03-17 13:57:35,068] [INFO] Diretórios criados para tabela: products
[2026-03-17 13:57:35,069] [INFO] Diretórios criados para tabela: sellers
[2026-03-17 13:57:35,070] [INFO] === Bootstrap Local Concluído ===
[2026-03-17 13:57:35,071] 

In [2]:
import uuid, time
from pyspark.sql.functions import col, lit
from lakehouse_engine.config import get_notebook_parameters, get_root_path, BronzeConfig
from lakehouse_engine.spark import get_spark_session
from lakehouse_engine.utils import setup_logging
from lakehouse_engine.io import DataFrameReader, DataFrameWriter

In [3]:
#PROVISORIO SPARK LOCAL
import ipywidgets as widgets
from IPython.display import display

dataset_widget = widgets.Text(value='customers', description='Dataset:')

display(dataset_widget)

Text(value='customers', description='Dataset:')

In [4]:
layer = "bronze"

#Load Spark Session and get parameters Databricks/Local
spark = get_spark_session()
dataset = get_notebook_parameters(dataset_widget)

# Load logging
log = setup_logging(f"{dataset}_{layer}")

bronze_cfg = BronzeConfig(dataset, layer)
bronze_cfg.load_yaml()

[2026-03-17 13:57:36,443] [WARNING] [LOCAL] Running in local mode.


In [5]:
# Create variables from yaml
file_format = bronze_cfg.file_format
spark_options = bronze_cfg.spark_options
target_table = bronze_cfg.target_table
target_schema = bronze_cfg.target_schema
target_catalog = bronze_cfg.target_catalog

# Create logical variables
run_id = str(uuid.uuid4())
ingest_ts = spark.sql("select current_timestamp() as ts").collect()[0]["ts"]

# Create paths
root_path = get_root_path()
file_path = f"{root_path}/{target_catalog}/landing/files/{dataset}/"
checkpoint_path = f"{root_path}/{target_catalog}/landing/checkpoints/{dataset}/"

In [6]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")
log.info(f"[CTX] target_table={target_table} target_schema={target_schema} target_catalog={target_catalog}")
log.info(f"[CTX] file_path={file_path}")
log.info(f"[CTX] checkpoint_path={checkpoint_path}")
log.info(f"[CTX] ingest_ts={ingest_ts}")

start = time.time()
log.info(f"[BRONZE][START] run_id={run_id}")

#Bronze Layer: Read File → Add Ingest Metadata → Add Processing Metadata → Write Delta

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)
    
    # Read file
    df_raw = reader.read_file(file_path, file_format, spark_options)
    count_raw = df_raw.count()
    log.info(f"[STEP 1] READ: File ingested: {count_raw} rows")
    
    # Add metadata columns
    df_ingest = bronze_cfg.add_metadata(df_raw, ingest_ts)
    log.info(f"[STEP 2] ADD_METADATA: Added _ingest_ts and source metadata columns")
    
    # Write table
    writer.write_delta_batch(df_ingest, f"{target_schema}.{target_table}", "overwrite")
    count_final = df_ingest.count()
    log.info(f"[STEP 3] WRITE: Delta table written: {target_schema}.{target_table} with {count_final} rows")
    
    duration_s = round(time.time() - start, 3)
    log.info(f"[BRONZE][SUCCESS] run_id={run_id} dataset={dataset} total_rows={count_final} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[BRONZE][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise

[2026-03-17 13:57:38,561] [INFO] [PARAMS] dataset=customers layer=bronze
[2026-03-17 13:57:38,563] [INFO] [CTX] target_table=customers target_schema=bronze target_catalog=olist
[2026-03-17 13:57:38,564] [INFO] [CTX] file_path=data/olist/landing/files/customers/
[2026-03-17 13:57:38,566] [INFO] [CTX] checkpoint_path=data/olist/landing/checkpoints/customers/
[2026-03-17 13:57:38,568] [INFO] [CTX] ingest_ts=2026-03-17 13:57:36.913913
[2026-03-17 13:57:38,569] [INFO] [BRONZE][START] run_id=d52afbbe-a947-4174-8928-fa0e873b6183
[2026-03-17 13:57:40,984] [INFO] [STEP 1] READ: File ingested: 99441 rows
[2026-03-17 13:57:41,119] [INFO] [STEP 2] ADD_METADATA: Added _ingest_ts and source metadata columns
[2026-03-17 13:57:50,762] [INFO] [STEP 3] WRITE: Delta table written: bronze.customers with 99441 rows
[2026-03-17 13:57:50,765] [INFO] [BRONZE][SUCCESS] run_id=d52afbbe-a947-4174-8928-fa0e873b6183 dataset=customers total_rows=99441 duration_s=12.196s
